# Train a Khiops predictor from Parquet

Select the project's `.venv` kernel and run all cells. This example creates a small local Parquet file; no download is needed.

In [ ]:
from pathlib import Path

import duckdb

from khiops_duckdb import train_predictor

sandbox = Path.cwd() if Path.cwd().name == "sandbox" else Path.cwd() / "sandbox"
sandbox.mkdir(exist_ok=True)
parquet_path = sandbox / "sample_customers.parquet"

## Open a Parquet file

This cell creates and previews sample data. To use your own file, set `parquet_path` above and skip this cell. Change `target="churn"` below to your target column.

In [5]:
with duckdb.connect(config={"threads": 2}) as con:
    con.sql("""
        SELECT 'c' || i::VARCHAR AS customer_id,
               (i % 2)::DOUBLE AS usage,
               CASE WHEN i % 2 = 0 THEN 'stay' ELSE 'leave' END AS churn
        FROM range(200) AS customers(i)
    """).write_parquet(str(parquet_path))
    preview = con.read_parquet(str(parquet_path)).limit(5).df()
preview

,customer_id,usage,churn
0,c0,0.0,stay
1,c1,1.0,leave
2,c2,0.0,stay
3,c3,1.0,leave
4,c4,0.0,stay


## Train

Pass the Parquet path directly to Khiops. The report and model stay in `parquet_results/`. One core and no trees keep this example quick.

In [6]:
report_path, model_path = train_predictor(
    parquet_path,
    target="churn",
    output_dir=sandbox / "parquet_results",
    max_trees=0,
    max_cores=1,
)
assert Path(report_path).is_file()
assert model_path is not None and Path(model_path).is_file()
{"report": report_path, "model": model_path}

/home/elouen/python/khiops-duckdb/.venv/lib/python3.12/site-packages/khiops/core/internals/runner.py:1136: UserWarning: Sample datasets location does not exist (/home/elouen/khiops_data/samples). Execute the kh-download-datasets script or the khiops.tools.download_datasets function to download them.
  self._initialize_default_samples_dir()


{'report': '/home/elouen/python/khiops-duckdb/sandbox/parquet_results/report.khj',
 'model': '/home/elouen/python/khiops-duckdb/sandbox/parquet_results/report.model.kdic'}